# Phase 2d — Post-training Diagnostics (Colab orchestrator)

Thin orchestration only; all logic lives in `src/analysis/`.

1. Mount Drive + restore the repo
2. Run `phase2_diagnostics` — extracts trained-encoder features and runs the **unchanged** Phase 1 diagnostics (FID/MMD/PCA) so numbers are comparable to the Phase 1 baseline (FID 240.65)
3. Display `phase2_report.md` (before/after, ablation, generalisation tables)
4. Plots: PCA scatter (coral_on vs coral_off) and the cross-dataset FID bar chart

Use a **GPU runtime** with Drive mounted — feature extraction over EyePACS (35,108) + OLIVES (3,142) is heavy.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

CUDA: True Tesla T4


In [ ]:
# Setup: mount Drive, restore repo
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

REPO_DIR = "/content/dr-dissertation"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} reset --hard origin/main
%cd {REPO_DIR}

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU — set Runtime → Change runtime type → GPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/dr-dissertation'...
remote: Enumerating objects: 300, done.
remote: Counting objects: 100% (300/300), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 300 (delta 148), reused 221 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (300/300), 1.55 MiB | 31.80 MiB/s, done.
Resolving deltas: 100% (148/148), done.
CUDA available: True
GPU: Tesla T4


In [ ]:
%cd /content/dr-dissertation

/content/dr-dissertation


## Run Phase 2d diagnostics

Extracts trained-encoder features for each arm × split (idempotent — cached on Drive), runs the unchanged Phase 1 diagnostics, and writes `phase2_report.md` + `phase2_results.json` to the reports dir.

In [ ]:
!python -m src.analysis.phase2_diagnostics --config configs/diagnostics.yaml


[1/4] coral_on/full: extracting trained features
[extract] coral_on/full: loading trained backbone (device=cuda)
[backbone] loaded 318/318 tensors from 'encoder.backbone.*' (clean strict load)
[extract] coral_on/full: run_name=coral_on epoch=13 best_metric=0.6148 w_coral=1.0
[extract] coral_on/full: EyePACS (all shards)
coral_on/full eyepacs eyepacs_shard_000.pt: 100% 5000/5000 [00:07<00:00, 661.25img/s, ips=86.6]
coral_on/full eyepacs eyepacs_shard_001.pt: 100% 5000/5000 [00:06<00:00, 760.45img/s, ips=692.9]
coral_on/full eyepacs eyepacs_shard_002.pt: 100% 5000/5000 [00:06<00:00, 754.05img/s, ips=697.5]
coral_on/full eyepacs eyepacs_shard_003.pt: 100% 5000/5000 [00:06<00:00, 751.08img/s, ips=653.7]
coral_on/full eyepacs eyepacs_shard_004.pt: 100% 5000/5000 [00:06<00:00, 743.56img/s, ips=662.6]
coral_on/full eyepacs eyepacs_shard_005.pt: 100% 5000/5000 [00:06<00:00, 739.33img/s, ips=653.3]
coral_on/full eyepacs eyepacs_shard_006.pt: 100% 5000/5000 [00:06<00:00, 730.49img/s, ips=643.8]

## Report

The three tables: before/after vs Phase 1, the with/without-CORAL ablation, and the held-out test generalisation check.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

REPORTS_DIR = Path('/content/drive/MyDrive/dissertation/reports')
display(Markdown((REPORTS_DIR / 'phase2_report.md').read_text(encoding='utf-8')))

## Plots

PCA scatter of the trained embeddings (coral_on vs coral_off) to visualise the gap closing, and a bar chart of cross-dataset FID across Phase 1 / coral_off / coral_on. PCA reuses `compute_pca_projection` from the diagnostics module.

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.analysis.distribution_diagnostics import compute_pca_projection

FEATURES_DIR = Path('/content/drive/MyDrive/dissertation/features')
REPORTS_DIR = Path('/content/drive/MyDrive/dissertation/reports')


def load_feats(arm, dataset, split='full'):
    blob = torch.load(
        FEATURES_DIR / f'phase2_{arm}_{split}_{dataset}_features.pt',
        map_location='cpu',
        weights_only=False,
    )
    feats = blob['features']
    return feats.numpy() if isinstance(feats, torch.Tensor) else np.asarray(feats)


# PCA scatter: coral_on vs coral_off (full data)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, arm in zip(axes, ['coral_on', 'coral_off']):
    eye = load_feats(arm, 'eyepacs')
    oli = load_feats(arm, 'olives')
    coords_eye, coords_oli, pca = compute_pca_projection(eye, oli)
    var = pca.explained_variance_ratio_
    ax.scatter(coords_eye[:, 0], coords_eye[:, 1], s=6, alpha=0.3, label='EyePACS')
    ax.scatter(coords_oli[:, 0], coords_oli[:, 1], s=6, alpha=0.3, label='OLIVES')
    ax.set_title(f'{arm} (PC1 {var[0]:.1%}, PC2 {var[1]:.1%})')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend()
fig.suptitle('Trained-encoder embeddings: EyePACS vs OLIVES')
plt.tight_layout()
plt.show()

# FID bar chart: Phase 1 baseline vs coral_off vs coral_on (full data)
with open(REPORTS_DIR / 'phase2_results.json') as fh:
    res = json.load(fh)
baseline_fid = res['phase1_baseline']['fid']
off_fid = res['results']['coral_off']['full']['fid']
on_fid = res['results']['coral_on']['full']['fid']
labels = ['Phase 1\n(pretrained)', 'coral_off\n(trained)', 'coral_on\n(trained)']
values = [baseline_fid, off_fid, on_fid]
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, values, color=['#888888', '#d62728', '#2ca02c'])
ax.axhline(100, ls='--', color='k', lw=1, label='target < 100')
ax.set_ylabel('Fréchet distance (FID)')
ax.set_title('Cross-dataset FID: before vs after alignment')
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.1f}', ha='center', va='bottom')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Force Colab to match GitHub exactly (we only run in Colab, never edit here)
!git -C /content/dr-dissertation fetch origin && git -C /content/dr-dissertation reset --hard origin/main
# Verify Colab is on the 4d commit before running (guards the wrong-commit trap)
!git -C /content/dr-dissertation log --oneline -1

# Mount Drive so the script can read the four phase JSONs and write PHASE4_RESULTS.md
from google.colab import drive; drive.mount('/content/drive')
# Assemble the consolidated report from the saved JSONs (no recomputation)
%cd /content/dr-dissertation
!python -m src.analysis.phase4_report --config configs/zero_shot.yaml

HEAD is now at 2f0a7de Phase 4d: consolidated Phase 4 report + figure pack
2f0a7de (HEAD -> main, origin/main, origin/HEAD) Phase 4d: consolidated Phase 4 report + figure pack
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/dr-dissertation
[1/3] introspecting inputs
[introspect] scanning report inputs in /content/drive/MyDrive/dissertation/reports
  phase3a_calibration.json     FOUND   n_images, accuracy, accuracy_ci95, qwk, qwk_ci95, per_class, calibration, selective_prediction
  phase4a_summary.json         FOUND   n_images, grade_distribution, n_distinct_grades, dominant_grade_fraction, confidence_maxprob, entropy_norm, flags, comparison
  phase4b_failure.json         FOUND   n_images, collapse, severity_signal, appearance, confidence_rescue, temporal, verdict
  phase4c_indomain.json        FOUND   biomarkers, cst, bcva, coverage, benchmarks
  phase3a_report.md            FOUND 
  phase4a_repor

In [ ]:
# Force Colab to match GitHub exactly (we only run in Colab, never edit here)
!git -C /content/dr-dissertation fetch origin && git -C /content/dr-dissertation reset --hard origin/main
# Verify Colab now has the corrected report commit
!git -C /content/dr-dissertation log --oneline -1

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 4.44 KiB | 4.44 MiB/s, done.
From https://github.com/savita10/dr-dissertation
   2f0a7de..4b67656  main       -> origin/main
HEAD is now at 4b67656 Phase 4d: corrected PHASE4_RESULTS.md (4b appearance-artifact framing; fluid biomarkers lead)
4b67656 (HEAD -> main, origin/main, origin/HEAD) Phase 4d: corrected PHASE4_RESULTS.md (4b appearance-artifact framing; fluid biomarkers lead)
